In [165]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

## Load database

In [166]:
df_patients = pd.read_csv("../database/patients.csv")
df_services_weekly = pd.read_csv("../database/services_weekly.csv")
df_staff_schedule = pd.read_csv("../database/staff_schedule.csv")
df_staff = pd.read_csv("../database/staff.csv")

print(f"Shape of df_patients: {df_patients.shape}")
print(f"Shape of df_services_weekly: {df_services_weekly.shape}")
print(f"Shape of df_staff_schedule: {df_staff_schedule.shape}")
print(f"Shape of df_staff: {df_staff.shape}")

Shape of df_patients: (1000, 7)
Shape of df_services_weekly: (208, 10)
Shape of df_staff_schedule: (6552, 6)
Shape of df_staff: (110, 4)


In [167]:
print(df_patients.head(2))
print(df_services_weekly.head(2))
print(df_staff_schedule.head(2))
print(df_staff.head(2))

     patient_id               name  age arrival_date departure_date  service  \
0  PAT-09484753  Richard Rodriguez   24   2025-03-16     2025-03-22  surgery   
1  PAT-f0644084     Shannon Walker    6   2025-12-13     2025-12-14  surgery   

   satisfaction  
0            61  
1            83  
   week  month    service  available_beds  patients_request  \
0     1      1  emergency              32                76   
1     1      1    surgery              45               130   

   patients_admitted  patients_refused  patient_satisfaction  staff_morale  \
0                 32                44                    67            70   
1                 45                85                    83            78   

  event  
0  none  
1   flu  
   week      staff_id    staff_name    role    service  present
0     1  STF-b77cdc60  Allison Hill  doctor  emergency        1
1     2  STF-b77cdc60  Allison Hill  doctor  emergency        1
       staff_id    staff_name    role    service
0  STF-5c

## Merge database

In [168]:
df_patients['arrival_date'] = pd.to_datetime(df_patients['arrival_date'])
df_patients['departure_date'] = pd.to_datetime(df_patients['departure_date'])
df_patients['month'] = df_patients['arrival_date'].dt.month
df_patients['week'] = (df_patients['arrival_date'].dt.day - 1) // 7 + 1

# Aggregate staff schedule
df_staff_weekly = (
    df_staff_schedule
    .groupby(["week", "service"])
    .agg(
        total_staff=('staff_id', 'nunique'),
        staff_present=("present", "sum"),
        staff_absens=("present", lambda x: (x == 0).sum())
    )
    .reset_index()
)

# Add staff information
df_staff_weekly_detail = df_staff_schedule.merge(
    df_staff,
    on="staff_id",
    how="left",
    suffixes=("_schedule", "_master")
)

# Combine weekly service + staff information
df_service = df_services_weekly.merge(
    df_staff_weekly,
    on=["week", "service"],
    how="left"
)

df = df_patients.merge(
    df_service,
    on=["month", "week", "service"],
    how="left"
)

print("Final shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

Final shape: (1000, 19)

Columns:
['patient_id', 'name', 'age', 'arrival_date', 'departure_date', 'service', 'satisfaction', 'month', 'week', 'available_beds', 'patients_request', 'patients_admitted', 'patients_refused', 'patient_satisfaction', 'staff_morale', 'event', 'total_staff', 'staff_present', 'staff_absens']


## Fill missing value

In [169]:
# Create service statistics
service_stats = (
    df_services_weekly
    .groupby("service")
    .agg(
        available_beds=("available_beds", "mean"),
        patients_request=("patients_request", "mean"),
        patients_admitted=("patients_admitted", "mean"),
        patients_refused=("patients_refused", "mean"),
        patient_satisfaction=("patient_satisfaction", "mean"),
        staff_morale=("staff_morale", "mean")
    )
    .reset_index()
)

In [170]:
df = df.merge(
    service_stats,
    on="service",
    how="left",
    suffixes=("", "_service_avg")
)

# Fill missing values using service average
fill_mapping = {
    "available_beds": "available_beds_service_avg",
    "patients_request": "patients_request_service_avg",
    "patients_admitted": "patients_admitted_service_avg",
    "patients_refused": "patients_refused_service_avg",
    "patient_satisfaction": "patient_satisfaction_service_avg",
    "staff_morale": "staff_morale_service_avg"
}

for target_col, source_col in fill_mapping.items():
    df[target_col] = df[target_col].fillna(df[source_col])

df = df.drop(columns=list(fill_mapping.values()))

In [171]:
staff_stats = (
    df_staff_schedule
    .groupby("service")
    .agg(
        total_staff=("staff_id", "nunique"),
        staff_present=("present", "sum"),
        staff_absens=("present", lambda x: (x == 0).sum())
    )
    .reset_index()
)

df = df.merge(
    staff_stats,
    on="service",
    how="left",
    suffixes=("", "_service_avg")
)

In [ ]:
# Fill missing values for staff statistics using service average
df["total_staff"] = df["total_staff"].fillna(df["total_staff_service_avg"])
df["staff_present"] = df["staff_present"].fillna(df["staff_present_service_avg"])
df["staff_absens"] = df["staff_absens"].fillna(df["staff_absens_service_avg"])

df = df.drop(
    columns=[
        "total_staff_service_avg",
        "staff_present_service_avg",
        "staff_absens_service_avg"
    ]
)

In [173]:
# Round the values  to 2 decimal places
numeric_cols = [
    "available_beds",
    "patients_request",
    "patients_admitted",
    "patients_refused",
    "patient_satisfaction",
    "staff_morale",
    "total_staff",
    "staff_present",
    "staff_absens"
]

# Make available beds column into integer type
df["available_beds"] = df["available_beds"].astype(int)
df['total_staff'] = df['total_staff'].astype(int)

df[numeric_cols] = df[numeric_cols].round(2)

# Drop the "event" column if it exists
df = df.drop(columns=["event"])

In [174]:
df.head(2)

,patient_id,name,age,arrival_date,departure_date,service,satisfaction,month,week,available_beds,patients_request,patients_admitted,patients_refused,patient_satisfaction,staff_morale,total_staff,staff_present,staff_absens
0,PAT-09484753,Richard Rodriguez,24,2025-03-16,2025-03-22,surgery,61,3,3,37,43.1,32.42,10.67,79.27,72.63,25,783.0,517.0
1,PAT-f0644084,Shannon Walker,6,2025-12-13,2025-12-14,surgery,83,12,2,37,43.1,32.42,10.67,79.27,72.63,25,783.0,517.0
